<a href="https://colab.research.google.com/github/FedericoTacchi/ML-project/blob/main/ML_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install ucimlrepo -q

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

pd.set_option("display.max_columns", None) #show all the columns

In [10]:
bike = fetch_ucirepo(id=560) # 560 is dataset number in the URL of UCI
df = bike.data.original.copy() # the complete table, without the division  x/y of UCI
from google.colab import files
files.upload()
df = pd.read_csv("SeoulBikeData.csv", encoding="latin1")

Saving SeoulBikeData.csv to SeoulBikeData.csv


In [11]:
print(df.shape) #(raws, columns)
df.head() #first 5 raws
df.info() #every non-null type of column and values
df.describe().T #mean, min, max, quartiles (T = transposed)
#`shape` tells you how many observations and variables you have.
#`info()` shows the data type of each column: if `Date` appears as
#an `object`, it means it is text, not an actual date.
#`describe()` gives you an idea of ​​the scales—for example,
#how much the temperature varies,
#or how high the maximum number of rentals is compared to the average.

(8760, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       8760 non-null   object 
 1   Rented Bike Count          8760 non-null   int64  
 2   Hour                       8760 non-null   int64  
 3   Temperature(°C)            8760 non-null   float64
 4   Humidity(%)                8760 non-null   int64  
 5   Wind speed (m/s)           8760 non-null   float64
 6   Visibility (10m)           8760 non-null   int64  
 7   Dew point temperature(°C)  8760 non-null   float64
 8   Solar Radiation (MJ/m2)    8760 non-null   float64
 9   Rainfall(mm)               8760 non-null   float64
 10  Snowfall (cm)              8760 non-null   float64
 11  Seasons                    8760 non-null   object 
 12  Holiday                    8760 non-null   object 
 13  Functioning Day            8760 non-n

,count,mean,std,min,25%,50%,75%,max
Rented Bike Count,8760.0,704.602055,644.997468,0.0,191.00,504.50,1065.25,3556.00
Hour,8760.0,11.500000,6.922582,0.0,5.75,11.50,17.25,23.00
Temperature(°C),8760.0,12.882922,11.944825,-17.8,3.50,13.70,22.50,39.40
Humidity(%),8760.0,58.226256,20.362413,0.0,42.00,57.00,74.00,98.00
Wind speed (m/s),8760.0,1.724909,1.036300,0.0,0.90,1.50,2.30,7.40
Visibility (10m),8760.0,1436.825799,608.298712,27.0,940.00,1698.00,2000.00,2000.00
Dew point temperature(°C),8760.0,4.073813,13.060369,-30.6,-4.70,5.10,14.80,27.20
Solar Radiation (MJ/m2),8760.0,0.569111,0.868746,0.0,0.00,0.01,0.93,3.52
Rainfall(mm),8760.0,0.148687,1.128193,0.0,0.00,0.00,0.00,35.00
Snowfall (cm),8760.0,0.075068,0.436746,0.0,0.00,0.00,0.00,8.80


In [12]:
TARGET = "Rented Bike Count"
FUNC = "Functioning Day"

print(df.columns.tolist())               # exact column names
print("Expected raws 365*24:", 365 * 24)
print("Total missing values:", df.isna().sum().sum())
print("Duplicated raws:", df.duplicated().sum())

print(df["Date"].sample(8, random_state=0))
print(df["Hour"].unique())
print(df["Seasons"].unique())
print(df["Holiday"].unique())

print(df[FUNC].value_counts())                # how many 'non-functioning' hours
print(df.groupby(FUNC)[TARGET].describe())    # how many rentals per functioning and non-functioning hours
#value_counts() tells how many non-operative hours are present,
#and groupby shows if in those hours 'Rented Bike Count' is 0. If
#it is 0 then remouve those rows make sense.


['Date', 'Rented Bike Count', 'Hour', 'Temperature(°C)', 'Humidity(%)', 'Wind speed (m/s)', 'Visibility (10m)', 'Dew point temperature(°C)', 'Solar Radiation (MJ/m2)', 'Rainfall(mm)', 'Snowfall (cm)', 'Seasons', 'Holiday', 'Functioning Day']
Expected raws 365*24: 8760
Total missing values: 0
Duplicated raws: 0
4059    19/05/2018
6914    15/09/2018
233     10/12/2017
982     10/01/2018
5967    06/08/2018
2182    01/03/2018
3996    16/05/2018
1652    07/02/2018
Name: Date, dtype: object
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
['Winter' 'Spring' 'Summer' 'Autumn']
['No Holiday' 'Holiday']
Functioning Day
Yes    8465
No      295
Name: count, dtype: int64
                  count        mean         std  min    25%    50%     75%  \
Functioning Day                                                              
No                295.0    0.000000    0.000000  0.0    0.0    0.0     0.0   
Yes              8465.0  729.156999  642.351166  2.0  214.0  542.0  1084.

In [ ]:
#renaming columns and parsing of data
df = df.rename(columns={
    "Date": "date", "Hour": "hour", "Rented Bike Count": "bike_count",
    "Temperature(°C)": "temp", "Humidity(%)": "humidity",
    "Wind speed (m/s)": "wind", "Visibility (10m)": "visibility",
    "Dew point temperature(°C)": "dew_point", "Solar Radiation (MJ/m2)": "solar",
    "Rainfall(mm)": "rain", "Snowfall (cm)": "snow",
    "Seasons": "season", "Holiday": "holiday", "Functioning Day": "functioning",
})
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")
print(df["date"].min(), df["date"].max())
#'rename' gives short names and without space. 'to_datetime' with 'format' change text
#in real dates, and the explicit format prevents pandas to shift day and month.
